# Local portfolio edition
The analysis is preserved from the original submitted notebook. Only data paths and inline installation cells were adjusted for local use; saved outputs are cleared. Launch Jupyter from the repository root after installing requirements.txt. This copy has not been run end to end.


# **Data Exploration**

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
dataset = pd.read_csv('COMP1801_Coursework_Dataset.csv')

In [ ]:
dataset.shape

In [ ]:
dataset.head()

In [ ]:
dataset.info()

In [ ]:
dataset.describe()

In [ ]:
dataset.isnull().sum()

In [ ]:
# Lifespan Distribution
plt.figure(figsize=(10, 5))
sns.histplot(dataset['Lifespan'], kde=True, color='blue', bins=30)
plt.title('Distribution of Lifespan')
plt.xlabel('Lifespan')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Boxplots for Cooling Rate
plt.figure(figsize=(8, 5))
sns.boxplot(data=dataset, x='coolingRate', color='lightcoral')
plt.title('Cooling Rate Distribution')
plt.xlabel('Cooling Rate')
plt.show()

# Boxplots for Heat Treat Time
plt.figure(figsize=(8, 5))
sns.boxplot(data=dataset, x='HeatTreatTime', color='gold')
plt.title('Heat Treat Time Distribution')
plt.xlabel('Heat Treat Time')
plt.show()

# Boxplots for Nickel Percentage
plt.figure(figsize=(8, 5))
sns.boxplot(data=dataset, x='Nickel%', color='lightgreen')
plt.title('Nickel% Distribution')
plt.xlabel('Nickel%')
plt.show()

# Boxplots for Small Defects
plt.figure(figsize=(8, 5))
sns.boxplot(data=dataset, x='smallDefects', color='lightblue')
plt.title('Small Defects Distribution')
plt.xlabel('Small Defects')
plt.show()

# Boxplots for Iron
plt.figure(figsize=(8, 5))
sns.boxplot(data=dataset, x='Iron%', color='blue')
plt.title('Iron')
plt.xlabel('Iron')
plt.show()

In [ ]:
# Correlation Matrix
numeric_dataset = dataset.select_dtypes(include=['number'])
plt.figure(figsize=(10, 6))
sns.heatmap(numeric_dataset.corr(), cbar=True, square=True, fmt='.2f', annot=True, annot_kws={'size':8}, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

# **Regression Implementation**

### Linear Regression

**With Categorical Feature**

In [ ]:
# Define features and target variable
categorical_features = ['partType', 'microstructure', 'seedLocation', 'castType']
numerical_features = ['coolingRate', 'quenchTime', 'HeatTreatTime', 'Nickel%', 'Iron%', 'Chromium%', 'smallDefects']
target = 'Lifespan'

In [ ]:
# Separate features and target
X = dataset[categorical_features + numerical_features]
y = dataset[target]

**With One Hot Encoding**

In [ ]:
# One-hot encoding for categorical features, standard scaling for numerical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)  # Updated argument
    ]
)

# Train-Test-Validation Split (80-10-10)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

X_train_preprocessed = preprocessor.fit_transform(X_train)
X_val_preprocessed = preprocessor.transform(X_val)
X_test_preprocessed = preprocessor.transform(X_test)

# Polynomial Regression Experimentation
degrees = [1, 2, 3, 4]  # Degrees to test
results = []

for degree in degrees:
    # Polynomial Features
    poly = PolynomialFeatures(degree=degree)
    X_train_poly = poly.fit_transform(X_train_preprocessed)
    X_val_poly = poly.transform(X_val_preprocessed)
    X_test_poly = poly.transform(X_test_preprocessed)

    # Train Linear Regression Model
    linear_model = LinearRegression()
    linear_model.fit(X_train_poly, y_train)

    # Evaluate on Validation Set
    y_val_pred = linear_model.predict(X_val_poly)
    val_mae = mean_absolute_error(y_val, y_val_pred)
    val_mse = mean_squared_error(y_val, y_val_pred)
    val_r2 = r2_score(y_val, y_val_pred)

    # Store Validation Results
    results.append({
        'Degree': degree,
        'Validation MAE': val_mae,
        'Validation MSE': val_mse,
        'Validation R²': val_r2
    })

results_df = pd.DataFrame(results)
print("\nValidation Results for Polynomial Regression:")
print(results_df)

# Select the best degree based on Validation MSE
best_degree = results_df.loc[results_df['Validation MSE'].idxmin(), 'Degree']
print(f"\nBest Degree for Polynomial Features: {best_degree}")

# Train Final Model on Combined Training + Validation Set
X_train_val_preprocessed = np.vstack((X_train_preprocessed, X_val_preprocessed))
y_train_val = np.concatenate((y_train, y_val))

poly = PolynomialFeatures(degree=best_degree)
X_train_val_poly = poly.fit_transform(X_train_val_preprocessed)

linear_model = LinearRegression()
linear_model.fit(X_train_val_poly, y_train_val)

# Evaluate on the Test Set
X_test_poly = poly.transform(X_test_preprocessed)
y_test_pred = linear_model.predict(X_test_poly)

# Compute Test Set Metrics
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

# Print Test Set Performance
print("\nTest Set Performance for Best Polynomial Regression Model:")
print(f"MAE: {test_mae:.2f}")
print(f"MSE: {test_mse:.2f}")
print(f"R²: {test_r2:.2f}")

# Create a scatter plot for actual vs predicted values
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_test_pred, alpha=0.7, color='blue', label='Predictions')
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linestyle='--', label='Ideal Fit')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Actual vs Predicted Values for Best Polynomial Regression Model')
plt.legend()
plt.grid()
plt.show()




**Only Numeric Features**

In [ ]:
# Separate features and target
X = dataset[numerical_features]
y = dataset[target]

In [ ]:
# Train-Test-Validation Split (80-10-10)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Feature Scaling: Standardizing the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Polynomial Regression Experimentation
degrees = [1, 2, 3, 4]
results = []

for degree in degrees:
    # Polynomial Features
    poly = PolynomialFeatures(degree=degree)
    X_train_poly = poly.fit_transform(X_train_scaled)
    X_val_poly = poly.transform(X_val_scaled)
    X_test_poly = poly.transform(X_test_scaled)

    # Train Linear Regression Model
    linear_model = LinearRegression()
    linear_model.fit(X_train_poly, y_train)

    # Evaluate on Validation Set
    y_val_pred = linear_model.predict(X_val_poly)
    val_mae = mean_absolute_error(y_val, y_val_pred)
    val_mse = mean_squared_error(y_val, y_val_pred)
    val_r2 = r2_score(y_val, y_val_pred)

    # Store Validation Results
    results.append({
        'Degree': degree,
        'Validation MAE': val_mae,
        'Validation MSE': val_mse,
        'Validation R²': val_r2
    })

results_df = pd.DataFrame(results)
print("\nValidation Results for Polynomial Regression:")
print(results_df)

# Select the best degree based on Validation MSE
best_degree = results_df.loc[results_df['Validation MSE'].idxmin(), 'Degree']
print(f"\nBest Degree for Polynomial Features: {best_degree}")

# Train Final Model on Combined Training + Validation Set
X_train_val_scaled = np.vstack((X_train_scaled, X_val_scaled))
y_train_val = np.concatenate((y_train, y_val))

poly = PolynomialFeatures(degree=best_degree)
X_train_val_poly = poly.fit_transform(X_train_val_scaled)

linear_model = LinearRegression()
linear_model.fit(X_train_val_poly, y_train_val)

# Evaluate on the Test Set
X_test_poly = poly.transform(X_test_scaled)  # Transform the test set using the same fitted PolynomialFeatures
y_test_pred = linear_model.predict(X_test_poly)

# Compute Test Set Metrics
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

# Print Test Set Performance
print("\nTest Set Performance for Best Polynomial Regression Model:")
print(f"MAE: {test_mae:.2f}")
print(f"MSE: {test_mse:.2f}")
print(f"R²: {test_r2:.2f}")

# Create a scatter plot for actual vs predicted values
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_test_pred, alpha=0.7, color='blue', label='Predictions')
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linestyle='--', label='Ideal Fit')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Actual vs Predicted Values for Best Polynomial Regression Model')
plt.legend()
plt.grid()
plt.show()


# **ANN Model**

**With One Hot Encoding**

In [ ]:
# Define features and target variable
categorical_features = ['partType', 'microstructure', 'seedLocation', 'castType']
numerical_features = ['coolingRate', 'quenchTime', 'HeatTreatTime', 'Nickel%', 'Iron%', 'Chromium%', 'smallDefects']
target = 'Lifespan'

# Separate features and target
X = dataset[categorical_features + numerical_features]
y = dataset[target]

# One-hot encoding for categorical features, standard scaling for numerical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ]
)

In [ ]:
# Split the dataset into Train (80%) and Temp (20%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)

# Split the Temp set into Validation (10%) and Test (10%)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Scale the numerical features
X_train_preprocess = preprocessor.fit_transform(X_train)
X_val_preprocess = preprocessor.transform(X_val)
X_test_preprocess = preprocessor.transform(X_test)

# Define the hyper-parameter grid for ANN
mlp_params = [
    {'hidden_layer_sizes': (50,), 'activation': 'relu', 'learning_rate_init': 0.001, 'max_iter': 500},
    {'hidden_layer_sizes': (50,), 'activation': 'relu', 'learning_rate_init': 0.01, 'max_iter': 500},
    {'hidden_layer_sizes': (50,), 'activation': 'relu', 'learning_rate_init': 0.1, 'max_iter': 500},
    {'hidden_layer_sizes': (100,), 'activation': 'relu', 'learning_rate_init': 0.001, 'max_iter': 500},
    {'hidden_layer_sizes': (100,), 'activation': 'relu', 'learning_rate_init': 0.01, 'max_iter': 500},
    {'hidden_layer_sizes': (100,), 'activation': 'relu', 'learning_rate_init': 0.1, 'max_iter': 500},
    {'hidden_layer_sizes': (100,50), 'activation': 'relu', 'learning_rate_init': 0.001, 'max_iter': 500},
    {'hidden_layer_sizes': (100,50), 'activation': 'relu', 'learning_rate_init': 0.01, 'max_iter': 500},
    {'hidden_layer_sizes': (100,50), 'activation': 'relu', 'learning_rate_init': 0.1, 'max_iter': 500},
    {'hidden_layer_sizes': (50,), 'activation': 'relu', 'learning_rate_init': 0.001, 'max_iter': 250},
    {'hidden_layer_sizes': (50,), 'activation': 'relu', 'learning_rate_init': 0.01, 'max_iter': 250},
    {'hidden_layer_sizes': (50,), 'activation': 'relu', 'learning_rate_init': 0.1, 'max_iter': 250},
    {'hidden_layer_sizes': (100,), 'activation': 'relu', 'learning_rate_init': 0.001, 'max_iter': 250},
    {'hidden_layer_sizes': (100,), 'activation': 'relu', 'learning_rate_init': 0.01, 'max_iter': 250},
    {'hidden_layer_sizes': (100,), 'activation': 'relu', 'learning_rate_init': 0.1, 'max_iter': 250},
    {'hidden_layer_sizes': (100,50), 'activation': 'relu', 'learning_rate_init': 0.001, 'max_iter': 250},
    {'hidden_layer_sizes': (100,50), 'activation': 'relu', 'learning_rate_init': 0.01, 'max_iter': 250},
    {'hidden_layer_sizes': (100,50), 'activation': 'relu', 'learning_rate_init': 0.1, 'max_iter': 250},
    {'hidden_layer_sizes': (50,), 'activation': 'tanh', 'learning_rate_init': 0.001, 'max_iter': 500},
    {'hidden_layer_sizes': (50,), 'activation': 'tanh', 'learning_rate_init': 0.01, 'max_iter': 500},
    {'hidden_layer_sizes': (50,), 'activation': 'tanh', 'learning_rate_init': 0.1, 'max_iter': 500},
    {'hidden_layer_sizes': (100,), 'activation': 'tanh', 'learning_rate_init': 0.001, 'max_iter': 500},
    {'hidden_layer_sizes': (100,), 'activation': 'tanh', 'learning_rate_init': 0.01, 'max_iter': 500},
    {'hidden_layer_sizes': (100,), 'activation': 'tanh', 'learning_rate_init': 0.1, 'max_iter': 500},
    {'hidden_layer_sizes': (100,50), 'activation': 'tanh', 'learning_rate_init': 0.001, 'max_iter': 500},
    {'hidden_layer_sizes': (100,50), 'activation': 'tanh', 'learning_rate_init': 0.01, 'max_iter': 500},
    {'hidden_layer_sizes': (100,50), 'activation': 'tanh', 'learning_rate_init': 0.1, 'max_iter': 500},
    {'hidden_layer_sizes': (50,), 'activation': 'tanh', 'learning_rate_init': 0.001, 'max_iter': 250},
    {'hidden_layer_sizes': (50,), 'activation': 'tanh', 'learning_rate_init': 0.01, 'max_iter': 250},
    {'hidden_layer_sizes': (50,), 'activation': 'tanh', 'learning_rate_init': 0.1, 'max_iter': 250},
    {'hidden_layer_sizes': (100,), 'activation': 'tanh', 'learning_rate_init': 0.001, 'max_iter': 250},
    {'hidden_layer_sizes': (100,), 'activation': 'tanh', 'learning_rate_init': 0.01, 'max_iter': 250},
    {'hidden_layer_sizes': (100,), 'activation': 'tanh', 'learning_rate_init': 0.1, 'max_iter': 250},
    {'hidden_layer_sizes': (100,50), 'activation': 'tanh', 'learning_rate_init': 0.001, 'max_iter': 250},
    {'hidden_layer_sizes': (100,50), 'activation': 'tanh', 'learning_rate_init': 0.01, 'max_iter': 250},
    {'hidden_layer_sizes': (100,50), 'activation': 'tanh', 'learning_rate_init': 0.1, 'max_iter': 250},
]

# Tune hyperparameters manually using the Validation set
results = []
for params in mlp_params:
    print(f"Training with parameters: {params}")
    model = MLPRegressor(
        hidden_layer_sizes=params['hidden_layer_sizes'],
        activation=params['activation'],
        learning_rate_init=params['learning_rate_init'],
        max_iter=params['max_iter'],
        random_state=42
    )
    model.fit(X_train_preprocess, y_train)
    y_val_pred = model.predict(X_val_preprocess)

    # Calculate metrics on Validation set
    mae = mean_absolute_error(y_val, y_val_pred)
    mse = mean_squared_error(y_val, y_val_pred)
    r2 = r2_score(y_val, y_val_pred)

    # Store results
    results.append({
        'Parameters': params,
        'MAE': mae,
        'MSE': mse,
        'R²': r2
    })


results_df = pd.DataFrame(results)
print(results_df)

best_params = results_df.loc[results_df['MSE'].idxmin(), 'Parameters']
print("\nBest Parameters Based on Validation MSE:")
print(best_params)

# Train the Best Model on the Combined Training and Validation Set
X_train_val = np.vstack((X_train_preprocess, X_val_preprocess))
y_train_val = np.concatenate((y_train, y_val))
best_model = MLPRegressor(
    hidden_layer_sizes=best_params['hidden_layer_sizes'],
    activation=best_params['activation'],
    learning_rate_init=best_params['learning_rate_init'],
    max_iter=best_params['max_iter'],
    random_state=42
)
best_model.fit(X_train_val, y_train_val)

# Evaluate the final model on the Test Set
y_test_pred = best_model.predict(X_test_preprocess)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

# Print Test Set Performance
print("\nTest Set Performance of the Best Model:")
print(f"MAE: {test_mae:.2f}")
print(f"MSE: {test_mse:.2f}")
print(f"R²: {test_r2:.2f}")

# Create a scatter plot for actual vs predicted values
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_test_pred, alpha=0.7, color='blue', label='Predictions')
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linestyle='--', label='Ideal Fit')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Actual vs Predicted Values for Best ANN Model')
plt.legend()
plt.grid(True)
plt.show()



**Numerical Features**

In [ ]:
# Separate features and target
X = dataset[numerical_features]
y = dataset[target]

In [ ]:
# Split the dataset into Train (80%) and Temp (20%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)

# Split the Temp set into Validation (10%) and Test (10%)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Scale the numerical features
scaler = StandardScaler()
X_train_scale = scaler.fit_transform(X_train)
X_val_scale = scaler.transform(X_val)
X_test_scale = scaler.transform(X_test)

# Define the hyper-parameter grid for ANN
mlp_params = [
    {'hidden_layer_sizes': (50,), 'activation': 'relu', 'learning_rate_init': 0.001, 'max_iter': 500},
    {'hidden_layer_sizes': (50,), 'activation': 'relu', 'learning_rate_init': 0.01, 'max_iter': 500},
    {'hidden_layer_sizes': (50,), 'activation': 'relu', 'learning_rate_init': 0.1, 'max_iter': 500},
    {'hidden_layer_sizes': (100,), 'activation': 'relu', 'learning_rate_init': 0.001, 'max_iter': 500},
    {'hidden_layer_sizes': (100,), 'activation': 'relu', 'learning_rate_init': 0.01, 'max_iter': 500},
    {'hidden_layer_sizes': (100,), 'activation': 'relu', 'learning_rate_init': 0.1, 'max_iter': 500},
    {'hidden_layer_sizes': (100,50), 'activation': 'relu', 'learning_rate_init': 0.001, 'max_iter': 500},
    {'hidden_layer_sizes': (100,50), 'activation': 'relu', 'learning_rate_init': 0.01, 'max_iter': 500},
    {'hidden_layer_sizes': (100,50), 'activation': 'relu', 'learning_rate_init': 0.1, 'max_iter': 500},
    {'hidden_layer_sizes': (50,), 'activation': 'relu', 'learning_rate_init': 0.001, 'max_iter': 250},
    {'hidden_layer_sizes': (50,), 'activation': 'relu', 'learning_rate_init': 0.01, 'max_iter': 250},
    {'hidden_layer_sizes': (50,), 'activation': 'relu', 'learning_rate_init': 0.1, 'max_iter': 250},
    {'hidden_layer_sizes': (100,), 'activation': 'relu', 'learning_rate_init': 0.001, 'max_iter': 250},
    {'hidden_layer_sizes': (100,), 'activation': 'relu', 'learning_rate_init': 0.01, 'max_iter': 250},
    {'hidden_layer_sizes': (100,), 'activation': 'relu', 'learning_rate_init': 0.1, 'max_iter': 250},
    {'hidden_layer_sizes': (100,50), 'activation': 'relu', 'learning_rate_init': 0.001, 'max_iter': 250},
    {'hidden_layer_sizes': (100,50), 'activation': 'relu', 'learning_rate_init': 0.01, 'max_iter': 250},
    {'hidden_layer_sizes': (100,50), 'activation': 'relu', 'learning_rate_init': 0.1, 'max_iter': 250},
    {'hidden_layer_sizes': (50,), 'activation': 'tanh', 'learning_rate_init': 0.001, 'max_iter': 500},
    {'hidden_layer_sizes': (50,), 'activation': 'tanh', 'learning_rate_init': 0.01, 'max_iter': 500},
    {'hidden_layer_sizes': (50,), 'activation': 'tanh', 'learning_rate_init': 0.1, 'max_iter': 500},
    {'hidden_layer_sizes': (100,), 'activation': 'tanh', 'learning_rate_init': 0.001, 'max_iter': 500},
    {'hidden_layer_sizes': (100,), 'activation': 'tanh', 'learning_rate_init': 0.01, 'max_iter': 500},
    {'hidden_layer_sizes': (100,), 'activation': 'tanh', 'learning_rate_init': 0.1, 'max_iter': 500},
    {'hidden_layer_sizes': (100,50), 'activation': 'tanh', 'learning_rate_init': 0.001, 'max_iter': 500},
    {'hidden_layer_sizes': (100,50), 'activation': 'tanh', 'learning_rate_init': 0.01, 'max_iter': 500},
    {'hidden_layer_sizes': (100,50), 'activation': 'tanh', 'learning_rate_init': 0.1, 'max_iter': 500},
    {'hidden_layer_sizes': (50,), 'activation': 'tanh', 'learning_rate_init': 0.001, 'max_iter': 250},
    {'hidden_layer_sizes': (50,), 'activation': 'tanh', 'learning_rate_init': 0.01, 'max_iter': 250},
    {'hidden_layer_sizes': (50,), 'activation': 'tanh', 'learning_rate_init': 0.1, 'max_iter': 250},
    {'hidden_layer_sizes': (100,), 'activation': 'tanh', 'learning_rate_init': 0.001, 'max_iter': 250},
    {'hidden_layer_sizes': (100,), 'activation': 'tanh', 'learning_rate_init': 0.01, 'max_iter': 250},
    {'hidden_layer_sizes': (100,), 'activation': 'tanh', 'learning_rate_init': 0.1, 'max_iter': 250},
    {'hidden_layer_sizes': (100,50), 'activation': 'tanh', 'learning_rate_init': 0.001, 'max_iter': 250},
    {'hidden_layer_sizes': (100,50), 'activation': 'tanh', 'learning_rate_init': 0.01, 'max_iter': 250},
    {'hidden_layer_sizes': (100,50), 'activation': 'tanh', 'learning_rate_init': 0.1, 'max_iter': 250},
]

# Tune hyperparameters manually using the Validation set
results = []
for params in mlp_params:
    print(f"Training with parameters: {params}")
    model = MLPRegressor(
        hidden_layer_sizes=params['hidden_layer_sizes'],
        activation=params['activation'],
        learning_rate_init=params['learning_rate_init'],
        max_iter=params['max_iter'],
        random_state=42
    )
    model.fit(X_train_scale, y_train)
    y_val_pred = model.predict(X_val_scale)

    # Calculate metrics on Validation set
    mae = mean_absolute_error(y_val, y_val_pred)
    mse = mean_squared_error(y_val, y_val_pred)
    r2 = r2_score(y_val, y_val_pred)

    # Store results
    results.append({
        'Parameters': params,
        'MAE': mae,
        'MSE': mse,
        'R²': r2
    })


results_df = pd.DataFrame(results)
print(results_df)


best_params = results_df.loc[results_df['MSE'].idxmin(), 'Parameters']
print("\nBest Parameters Based on Validation MSE:")
print(best_params)

# Train the Best Model on the Combined Training and Validation Set
X_train_val = np.vstack((X_train_scale, X_val_scale))
y_train_val = np.concatenate((y_train, y_val))
best_model = MLPRegressor(
    hidden_layer_sizes=best_params['hidden_layer_sizes'],
    activation=best_params['activation'],
    learning_rate_init=best_params['learning_rate_init'],
    max_iter=best_params['max_iter'],
    random_state=42
)
best_model.fit(X_train_val, y_train_val)

# Evaluate the final model on the Test Set
y_test_pred = best_model.predict(X_test_scale)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

# Print Test Set Performance
print("\nTest Set Performance of the Best Model:")
print(f"MAE: {test_mae:.2f}")
print(f"MSE: {test_mse:.2f}")
print(f"R²: {test_r2:.2f}")

# Create a scatter plot for actual vs predicted values
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_test_pred, alpha=0.7, color='blue', label='Predictions')
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linestyle='--', label='Ideal Fit')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Actual vs Predicted Values for Best ANN Model')
plt.legend()
plt.grid(True)
plt.show()


# **Classification Implementation**

In [ ]:
dataset = pd.read_csv('COMP1801_Coursework_Dataset.csv')

# Exclude Lifespan column for clustering input
numerical_data = dataset[numerical_features]

# Standardize numerical features
scaler = StandardScaler()
numerical_data_scaled = scaler.fit_transform(numerical_data)

# Apply K-Means clustering
k = 3  #
kmeans = KMeans(n_clusters=k, random_state=42)

# Fit K-Means on scaled numerical features
dataset['cluster_labels'] = kmeans.fit_predict(numerical_data_scaled)

# Check the distribution of clusters
cluster_distribution = dataset['cluster_labels'].value_counts()
print("Cluster Distribution:")
print(cluster_distribution)

plt.figure(figsize=(6, 4))
cluster_distribution.plot(kind='bar', color='green')
plt.title('Cluster Distribution (Multi-Class Without Lifespan)')
plt.xlabel('Cluster')
plt.ylabel('Count')
plt.grid(axis='y')
plt.show()


In [ ]:
# Elbow Method to find the optimal k
wcss = []
# Test for k = 1 to 10
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=42)
    kmeans.fit(X_train_preprocessed)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 6))
plt.plot(range(1, 11), wcss, marker='o', color='b')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS')
plt.show()


In [ ]:
# Use the cluster labels as a new output class for multi-class classification
dataset['multi_class_labels'] = dataset['cluster_labels']

# Checking the distribution of multi-class labels
print("\nMulti-Class Label Distribution:")
print(dataset['multi_class_labels'].value_counts())


In [ ]:
# Add binary labels based on the 1500-hour threshold
dataset['1500_labels'] = (dataset['Lifespan'] >= 1500).astype(int)

# Define features and target for binary classification
X = dataset.drop(['Lifespan', 'multi_class_labels'], axis=1)
y_binary = dataset['1500_labels']

# Train-Test-Validation Split (80-10-10)
X_train, X_temp, y_train, y_temp = train_test_split(X, y_binary, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Define preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),  # Scale numerical features
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)  # Encode categorical features
    ]
)

# Apply preprocessing
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_val_preprocessed = preprocessor.transform(X_val)
X_test_preprocessed = preprocessor.transform(X_test)


In [ ]:
# Adjusted hyperparameter grid for Logistic Regression
log_reg_params = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear', 'saga', 'lbfgs'],
}

log_reg = LogisticRegression(max_iter=500, random_state=42)
log_reg_grid = GridSearchCV(estimator=log_reg, param_grid=log_reg_params, scoring='accuracy', cv=5, verbose=1, error_score='raise')
log_reg_grid.fit(X_train_preprocessed, y_train)

# Logistic Regression Hyperparameter Results
log_reg_results = pd.DataFrame(log_reg_grid.cv_results_)
log_reg_results_summary = log_reg_results[
    ['param_C', 'param_solver', 'mean_test_score', 'std_test_score']
].copy()
log_reg_results_summary.rename(columns={
    'param_C': 'C',
    'param_solver': 'Solver',
    'mean_test_score': 'Mean Accuracy (CV)',
    'std_test_score': 'Std Dev (CV)'
}, inplace=True)

print("\nLogistic Regression Hyperparameter Tuning Results:")
print(log_reg_results_summary)

In [ ]:
# ANN: Hyperparameter Tuning
ann_params = {
    'hidden_layer_sizes': [(50,), (100,), (100, 50)],
    'activation': ['relu', 'tanh'],
    'learning_rate_init': [0.001, 0.01],
    'max_iter': [500, 1000]
}

ann_model = MLPClassifier(random_state=42)
ann_grid = GridSearchCV(estimator=ann_model, param_grid=ann_params, scoring='accuracy', cv=5, verbose=1)
ann_grid.fit(X_train_preprocessed, y_train)

# ANN Hyperparameter Results
ann_results = pd.DataFrame(ann_grid.cv_results_)
ann_results_summary = ann_results[
    ['param_hidden_layer_sizes', 'param_activation', 'param_learning_rate_init', 'param_max_iter', 'mean_test_score', 'std_test_score']
].copy()
ann_results_summary.rename(columns={
    'param_hidden_layer_sizes': 'Hidden Layer Sizes',
    'param_activation': 'Activation Function',
    'param_learning_rate_init': 'Learning Rate',
    'param_max_iter': 'Max Iterations',
    'mean_test_score': 'Mean Accuracy (CV)',
    'std_test_score': 'Std Dev (CV)'
}, inplace=True)

print("\nANN Hyperparameter Tuning Results:")
print(ann_results_summary)


In [ ]:
# Get predictions from the trained Logistic Regression model
y_pred_log_reg = log_reg_grid.predict(X_test_preprocessed)
# Get predictions from the trained ANN model
y_pred_ann = ann_grid.predict(X_test_preprocessed)

print("\nLogistic Regression Classification Report:")
print(classification_report(y_test, y_pred_log_reg))

# Evaluation: ANN
print("\nANN Classification Report:")
print(classification_report(y_test, y_pred_ann))

# Confusion Matrix for Logistic Regression
cm_log_reg = confusion_matrix(y_test, y_pred_log_reg)
disp_log_reg = ConfusionMatrixDisplay(confusion_matrix=cm_log_reg, display_labels=np.unique(y_test))
disp_log_reg.plot(cmap='Blues')
plt.title("Confusion Matrix: Logistic Regression")
plt.show()

# Confusion Matrix for ANN
cm_ann = confusion_matrix(y_test, y_pred_ann)
disp_ann = ConfusionMatrixDisplay(confusion_matrix=cm_ann, display_labels=np.unique(y_test))
disp_ann.plot(cmap='Greens')
plt.title("Confusion Matrix: ANN")
plt.show()